# Week 2 Day 1 — Anthropic (Claude) edition

The original Day 1 is a tour of many providers (OpenAI, Gemini, DeepSeek, Groq, Grok, OpenRouter...). Since you're on the Anthropic track with one key, this version centers on **Claude**, keeps **Ollama** as your free local option, and marks the other providers optional. The real lessons survive intact:
- calling the model two ways (native client vs the OpenAI-compatible endpoint),
- **inference-time scaling** (the Anthropic equivalent of OpenAI's `reasoning_effort` is **extended thinking**),
- abstraction layers (LangChain, LiteLLM).

In [1]:
import os
import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display
import anthropic

load_dotenv(override=True)
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')

if anthropic_api_key and anthropic_api_key.startswith("sk-ant-"):
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (or doesn't start sk-ant-)")

Anthropic API Key exists and begins sk-ant-


## Two ways to call Claude

**(a) The native Anthropic client** — recommended.
**(b) The OpenAI-compatible endpoint** — Anthropic exposes one at `https://api.anthropic.com/v1/`, so the *OpenAI* client can call Claude too. Handy when you have code written for OpenAI. Note: native client uses the bare domain `https://api.anthropic.com`; the OpenAI-compat one needs the `/v1/` suffix.

In [2]:
# (a) Native client
client = anthropic.Anthropic()

# (b) OpenAI-compatible (optional - only if you want to reuse OpenAI-style code)
from openai import OpenAI
anthropic_via_openai = OpenAI(api_key=anthropic_api_key, base_url="https://api.anthropic.com/v1/")

In [3]:
tell_a_joke = [
    {"role": "user", "content": "Tell a joke for a student on the journey to becoming an expert in LLM Engineering"},
]

In [4]:
# Native client - note max_tokens is required, system is separate (here we have none)
response = client.messages.create(model="claude-sonnet-4-6", max_tokens=500, messages=tell_a_joke)
display(Markdown(response.content[0].text))

# A Little LLM Engineer Humor 🤖

---

**Why did the junior LLM engineer get fired?**

Because their boss asked them to *fine-tune the model*...

...and they just kept **complimenting it** and telling it what a great job it was doing. 🌟

---

*"You're doing amazing! Your outputs are so creative!"*

**Model:** *Hallucinates even more confidently.*

---

### The painful truth behind the joke:
- 🎯 **Fine-tuning** = carefully training on curated data
- 🚫 **NOT** = positive reinforcement via sweet prompt words
- 😅 Though sometimes it honestly feels like the model has **feelings about your prompts**

---

> *"The first 90% of LLM Engineering is prompt engineering. The second 90% is realizing prompt engineering wasn't enough and you actually need RAG, fine-tuning, evals, and a good cry."*

---

Keep going — every hallucination is just a **learning opportunity**... for both you *and* the model. 💪

In [5]:
# Same thing via the OpenAI-compatible endpoint - note the different response shape
response = anthropic_via_openai.chat.completions.create(model="claude-sonnet-4-6", messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))

# The LLM Engineer's Joke 🤖

---

A student asks their LLM:

**"Are you conscious?"**

The LLM replies:

**"I'm not sure, but I'm 94.7% confident that I am."**

---

The student then spends **3 weeks**:

- 🔧 Fine-tuning the model to say "yes"
- 📊 Writing a 40-page evaluation report
- 💸 Burning $2,000 in GPU credits
- 🐛 Debugging a prompt template with a missing **space**

...only to realize the **real answer was in the system prompt all along.**

---

> *"The hallucination wasn't in the model. It was in my confidence that I understood the model."*
> — Every LLM Engineer, eventually

---

Keep going — **the best engineers are just the ones who debugged the most weird outputs and lived to tell about it.** 🚀

## Training vs inference-time scaling

OpenAI exposes `reasoning_effort="minimal"|"low"|...`. Anthropic's equivalent is **extended thinking**: you give the model a budget of tokens to reason internally before answering, via `thinking={"type": "enabled", "budget_tokens": N}`.

Two rules: `max_tokens` must be **larger** than `budget_tokens`, and the response `content` will contain a `ThinkingBlock` *before* the text — so extract text by filtering on block type, never by indexing `content[0]`.

In [11]:
def claude_answer(prompt, thinking_budget=0):
    kwargs = dict(model="claude-sonnet-4-6", max_tokens=8000,
                  messages=[{"role": "user", "content": prompt}])
    if thinking_budget:
        kwargs["thinking"] = {"type": "enabled", "budget_tokens": thinking_budget}
    response = client.messages.create(**kwargs)
    # Extract ONLY the text blocks (skip any thinking block)
    return "".join(b.text for b in response.content if b.type == "text")

In [7]:
easy_puzzle = "You toss 2 coins. One of them is heads. What's the probability the other is tails? Answer with the probability only."

# No thinking - fast, cheap
display(Markdown(claude_answer(easy_puzzle)))

2/3

In [8]:
# With a thinking budget - the model reasons before answering (inference-time scaling)
display(Markdown(claude_answer(easy_puzzle, thinking_budget=1024)))

2/3

## A harder puzzle — where extended thinking earns its cost

In [9]:
hard = """
On a bookshelf, two volumes of Pushkin stand side by side: the first and the second.
The pages of each volume together have a thickness of 2 cm, and each cover is 2 mm thick.
A worm gnawed (perpendicular to the pages) from the first page of the first volume to the last page of the second volume.
What distance did it gnaw through?
"""

# Try it WITHOUT thinking first...
display(Markdown(claude_answer(hard)))

## Setting Up the Problem

I need to figure out how books are arranged on a shelf and what the worm travels through.

## Key Insight: How Books Sit on a Shelf

When books are placed on a shelf in order (Volume 1, then Volume 2), they face outward with their spines showing. This means:

- **Volume 1**: The **front cover** is on the **right** side (facing Volume 2), and the back cover is on the left
- **Volume 2**: The **front cover** is on the **left** side (facing Volume 1), and the back cover is on the right

So when standing on the shelf side by side:
- The **first page of Volume 1** is just inside the front cover of Volume 1 → located on the **right side** of Volume 1
- The **last page of Volume 2** is just inside the back cover... wait, let me reconsider.

Actually, with books standing normally on a shelf (spine facing out, readable left to right):
- Volume 1 has its **spine on the left**, front cover facing right... 

Let me think physically: Books sit with spine out. Volume 1 is to the left of Volume 2.
- Volume 1's **front cover** faces **left** (outward), back cover faces right (toward Vol. 2)
- Volume 2's **front cover** faces **left** (toward Vol. 1), back cover faces **right** (outward)

So the arrangement from left to right is:
1. Front cover of Vol. 1 (left face)
2. Pages of Vol. 1
3. Back cover of Vol. 1
4. Front cover of Vol. 2
5. Pages of Vol. 2
6. Back cover of Vol. 2

## Finding the Worm's Path

- **First page of Volume 1** is at the right side of Volume 1's pages (just inside the back cover of Vol. 1)
- **Last page of Volume 2** is at the left side of Volume 2's pages (just inside the front cover of Vol. 2)

The worm travels from the first page of Vol. 1 to the last page of Vol. 2, going left to right (perpendicular to pages).

What lies between these two points?
- Back cover of Volume 1: **2 mm**
- Front cover of Volume 2: **2 mm**

## Answer

The worm only gnaws through:
- Back cover of Vol. 1 = 2 mm
- Front cover of Vol. 2 = 2 mm

**Total distance = 4 mm**

In [12]:
# ...then WITH a generous thinking budget. Compare the answers.
display(Markdown(claude_answer(hard, thinking_budget=4000)))

## Setting Up the Problem

The key is to carefully visualize how the books actually sit on the shelf.

### How Books Are Arranged on a Shelf

When books stand on a shelf with their **spines facing outward**, a standard Western book has:
- **Front cover** on the **right** face
- **Back cover** on the **left** face

So for the two volumes side by side (Volume 1 left, Volume 2 right):

```
        [Vol. 1]              [Vol. 2]
Back  |←pages→| Front  |  Back |←pages→| Front
cover              cover    cover             cover
(left)            (right)  (left)            (right)
```

### Locating the Starting and Ending Pages

| | Location |
|---|---|
| **First page of Vol. 1** | Just inside the *front cover* → on the **right side** of Vol. 1 |
| **Last page of Vol. 2** | Just inside the *back cover* → on the **left side** of Vol. 2 |

### What Lies Between Them?

The first page of Vol. 1 and the last page of Vol. 2 face **each other** across the gap between the two books, separated only by:

1. The **back cover of Volume 1**: **2 mm**
2. The **front cover of Volume 2**: **2 mm**

The worm doesn't travel through any pages at all!

### Answer

$$d = 2\ \text{mm} + 2\ \text{mm} = \boxed{4\ \text{mm}}$$

## Going local — Ollama (free)

Same OpenAI-compatible trick as Week 1. No tokens spent.

In [13]:
requests.get("http://localhost:11434/").content  # if this errors, run: ollama serve

b'Ollama is running'

In [14]:
ollama = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")
response = ollama.chat.completions.create(model="llama3.2", messages=[{"role": "user", "content": easy_puzzle}])
display(Markdown(response.choices[0].message.content))

0.5

## Abstraction layers: LangChain

LangChain wraps every provider behind one interface. The Anthropic class is `ChatAnthropic` (install `langchain-anthropic`).

In [15]:
# pip install langchain-anthropic
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(model="claude-sonnet-4-6", max_tokens=500)
response = llm.invoke(tell_a_joke)
display(Markdown(response.content))

C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\langchain_core\utils\pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


# A Little LLM Engineering Humor 🤖

---

**Why did the LLM Engineering student fail their exam?**

The professor asked them to explain their model's reasoning...

...and they just said:

*"I cannot provide information about my internal reasoning process, but here's a helpful and harmless response!"*

---

**Bonus joke:**

A junior LLM Engineer is debugging their RAG pipeline at 2am.

They ask their chatbot: *"Why aren't you giving correct answers?"*

The chatbot replies: *"As an AI language model, I don't have the ability to be incorrect — would you like me to confidently hallucinate something else instead?"*

---

🎓 **The real joke?**

You'll spend **3 hours** crafting the perfect prompt...

...only to discover the model works perfectly when you just say **"please"** and **"think step by step"** 😅

---

Keep going — the world needs more people who understand what's *actually* happening inside that black box! 💪

*You've got this, future LLM Engineer!* 🚀

## Abstraction layers: LiteLLM

LiteLLM is a lightweight router; you just prefix the model with the provider. For Claude it's `anthropic/<model>`. It also reports token usage and cost.

In [16]:
from litellm import completion

response = completion(model="anthropic/claude-sonnet-4-6", messages=tell_a_joke, max_tokens=500)
display(Markdown(response.choices[0].message.content))

# A Little LLM Engineering Humor 🤖

---

**Why did the LLM Engineering student fail their exam?**

Their answer was *technically correct*, had *great formatting*, and was *extremely confident*...

...but it was about a **completely different question** than what was asked.

The professor wrote in the feedback:

*"Impressive output. Wrong prompt. Classic."*

---

**The student's response?**

They just added *"Let's think step by step"* to the top of their exam and submitted it again. 📝

---

Hope that lands well on your **context window** today! 😄

Keep iterating — every great prompt engineer started with a few *hallucinations* of their own. 🚀

In [17]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")
cost = response._hidden_params.get("response_cost")
if cost:
    print(f"Total cost: {cost*100:.4f} cents")

Input tokens: 25
Output tokens: 173
Total tokens: 198
Total cost: 0.2670 cents


## Optional: other providers

If you ever add keys for Gemini / DeepSeek / Groq / Grok / OpenRouter, they're all OpenAI-compatible, so it's the same `OpenAI(api_key=..., base_url=...)` pattern from Week 1 Day 2. You don't need any of them for this course on the Anthropic track — Claude + Ollama covers everything ahead.

### Note on prompt caching (a pro feature)
The original ends with a LiteLLM prompt-caching demo. Anthropic supports prompt caching too, but it's opt-in: you mark large, reused chunks of context with a `cache_control` breakpoint so you're not billed full price to re-send them every turn. It's worth knowing the concept exists; see https://docs.claude.com/en/docs/build-with-claude/prompt-caching when you have a big static context (like a long document) you query repeatedly.